# Part 2 of the pipeline demo, generating idf from geojson

This notebook demonstrates how to use the EPSM pipeline to:
1. Filter the geojson from DTCC
2. Enrich the geojson
3. Convert into IDF

**Authors:** Aaron Qiyu Liu and Sanjay Somanath

In [11]:
# Import required modules
import sys
from pathlib import Path

# Add parent directory to path to import geojson_processor
sys.path.insert(0, str(Path.cwd().parent))

dtcc_output_path = Path.cwd() / "dtcc_output"

# Note: GeoJSONToIDFConverter import moved to Step 2 to speed up kernel startup
print("✓ Basic imports complete. GeoJSONToIDFConverter will be imported when needed.")

✓ Basic imports complete. GeoJSONToIDFConverter will be imported when needed.


## Step 1: Read the GeoJSON data

Load data from Step 1 (DTCC pipeline)

In [12]:
import json

# Load the GeoJSON file
geojson_path = dtcc_output_path / "city.geojson"

with open(geojson_path, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)


## Step 2: Initialize the GeoJSONToIDFConverter

The converter requires a working directory where it will save the output IDF files.

In [13]:
# Import the converter here (lazy loading to speed up kernel startup)
from geojson_processor.geojson_to_idf import GeoJSONToIDFConverter

# Create output directory for IDF files
idf_output_path = Path.cwd() / "idf_output"
idf_output_path.mkdir(parents=True, exist_ok=True)

# Initialize the converter
converter = GeoJSONToIDFConverter(work_dir=str(idf_output_path))
print(f"Converter initialized with work directory: {converter.work_dir}")

Converter initialized with work directory: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output


## Step 2: Filter Buildings

Before converting to IDF, you can filter buildings based on:
- **Height range**: Remove very short buildings (< 3m) or very tall buildings (> 100m)
- **Floor area**: Remove very small buildings (< 100 m²)

This helps clean the dataset and speeds up simulation.

In [14]:
# Filter buildings based on height and area criteria
filtered_geojson_path = converter.filter_buildings(
    geojson_path=geojson_path,
    filter_height_less_than=3,      # Remove buildings < 3m tall
    filter_height_greater_than=100,  # Remove buildings > 100m tall
    filter_area_less_than=100        # Remove buildings < 100 m²
)

print(f"Filtered GeoJSON saved to: {filtered_geojson_path}")

# Check how many buildings remain
with open(filtered_geojson_path, 'r', encoding='utf-8') as f:
    filtered_data = json.load(f)
print(f"Buildings after filtering: {len(filtered_data.get('features', []))}")

Filtered GeoJSON saved to: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_filtered.geojson
Buildings after filtering: 780


## Step 3: Enrich GeoJSON with Building Properties

Enrichment adds properties required by Dragonfly/Honeybee:
- Building ID and name
- Number of stories (calculated from height)
- Window-to-wall ratio
- Building type (simulation vs context shading)

**Simulation Bounds (Optional):**
If you specify simulation bounds, only buildings inside will be fully simulated. Buildings outside become "context shading" (affects shadows but aren't simulated internally), which significantly speeds up simulations for large areas.

In [15]:
# Enrich the filtered GeoJSON with simulation bounds
# Trollhättan city center (EPSG:3006 - Swedish coordinate system)
simulation_bounds = {
     'west': 337500,
     'south': 6467500,
     'east': 340500,    # 3km wide
     'north': 6470500   # 3km tall
}

enriched_geojson_path = converter.enrich_geojson(
    geojson_path=filtered_geojson_path,  # Use filtered data, or geojson_path for original
    simulation_bounds=simulation_bounds
)

print(f"Enriched GeoJSON saved to: {enriched_geojson_path}")

# Check how buildings are classified
with open(enriched_geojson_path, 'r', encoding='utf-8') as f:
    enriched_data = json.load(f)
    
sim_count = sum(1 for f in enriched_data['features'] 
               if f['properties'].get('building_status') == 'Building')
ctx_count = sum(1 for f in enriched_data['features'] 
               if f['properties'].get('building_status') == 'Existing')

print(f"\n📊 Building Classification:")
print(f"  → Buildings to simulate: {sim_count}")
print(f"  → Context shading only: {ctx_count}")
print(f"  → Total: {sim_count + ctx_count}")

Enriched GeoJSON saved to: c:\Users\qiyu\EPSM\epsm\backend\notebooks\idf_output\city_enriched.geojson

📊 Building Classification:
  → Buildings to simulate: 251
  → Context shading only: 529
  → Total: 780


In [16]:
# Select which GeoJSON to convert
# Uncomment the version you want to use:

#input_geojson = geojson_path                    # Option 1: Original data (244 buildings)
# input_geojson = filtered_geojson_path         # Option 2: Filtered data
input_geojson = enriched_geojson_path         # Option 3: Enriched (all simulated)
#input_geojson = enriched_with_bounds_path     # Option 4: RECOMMENDED (simulation bounds)

print(f"Selected input: {input_geojson.name}")
print(f"File exists: {input_geojson.exists()}")

# Check what you're about to convert
if input_geojson.exists():
    with open(input_geojson, 'r', encoding='utf-8') as f:
        data = json.load(f)
    total_features = len(data.get('features', []))
    print(f"Total buildings: {total_features}")
    
    # If enriched with bounds, show simulation vs context breakdown
    if 'building_status' in data['features'][0].get('properties', {}):
        sim_count = sum(1 for f in data['features'] 
                       if f['properties'].get('building_status') == 'Building')
        ctx_count = sum(1 for f in data['features'] 
                       if f['properties'].get('building_status') == 'Existing')
        print(f"  → Buildings to simulate: {sim_count}")
        print(f"  → Context shading: {ctx_count}")
else:
    print("⚠️  File not found! Run the filtering/enrichment steps first.")

Selected input: city_enriched.geojson
File exists: True
Total buildings: 780
  → Buildings to simulate: 251
  → Context shading: 529


## Step 3: Convert GeoJSON to IDF

Now we'll use the converter to generate IDF files from the GeoJSON data. The conversion includes:
- Building geometry from footprints
- Floor heights and building volumes
- Window assignments
- HVAC system templates
- Design day creation for sizing

**Key parameters:**
- `use_multiplier`: Use floor multipliers for repeated floors (faster simulation)
- `timestep`: Simulation timesteps per hour (1 = fastest, 6 = detailed)
- `do_zone_sizing` and `do_system_sizing`: Enable HVAC sizing calculations
- `winter_design_temp` and `summer_design_temp`: Design day temperatures for HVAC sizing

In [ ]:
# Convert GeoJSON to IDF
# Note: This may take some time depending on the number of buildings

# Choose which GeoJSON to convert:
# - geojson_path: Original data (all buildings)
# - filtered_geojson_path: Filtered by height/area
# - enriched_geojson_path: Filtered + enriched (all simulated)
# - enriched_with_bounds_path: Filtered + enriched with simulation bounds (RECOMMENDED)

# Use the enriched data if available, otherwise use original
#input_geojson = geojson_path  # Change this to use filtered/enriched versions
input_geojson = enriched_geojson_path  # Uncomment after running enrichment

idf_path, gbxml_path = converter.convert_to_idf(
    geojson_path=input_geojson,
    use_multiplier=True,          # Use floor multipliers for faster simulation
    timestep=1,                    # 1 timestep per hour (fastest)
    do_zone_sizing=False,          # Skip zone sizing for speed
    do_system_sizing=False,        # Skip system sizing for speed
    winter_design_temp=-10,        # Winter design temperature (°C)
    summer_design_temp=30,         # Summer design temperature (°C)
    all_polygons_to_buildings=True # Treat all polygons as buildings (needed for DTCC data)
)



: 